# 📘 부록: GAN으로 손글씨 숫자 만들기

**GAN(생성적 적대 신경망)이란**
- 학습 데이터와 닮은 새 데이터를 만드는 생성 모델입니다. generative adversarial network의 줄임말로, 두 신경망이 서로 겨루며(adversarial) 함께 학습합니다.
- 교안 01·02의 CNN은 그림을 받아 "이 숫자는 7"처럼 답을 고르는 분류 모델입니다. GAN의 생성자는 거꾸로 무작위 숫자 64개를 받아 숫자 그림 한 장을 만듭니다.

**두 신경망**
- **생성자(generator):** 잡음(무작위 숫자)을 받아 가짜 그림을 만듭니다. 판별자가 진짜로 판단하는 그림을 만드는 것이 목표입니다.
- **판별자(discriminator):** 그림 한 장을 받아 진짜(학습 데이터)인지 가짜(생성자가 만든 그림)인지 가립니다. 출력은 로짓(logit) 하나입니다.
- 두 신경망을 번갈아 갱신하면, 판별자가 가려내는 기준이 높아질수록 생성자의 그림도 진짜에 가까워집니다.


<img src="images/90_gan_overview.png" width="1050" alt="제목은 'GAN: 위조범(생성자)과 감별사(판별자)가 겨루며 함께 좋아집니다'입니다. 왼쪽에 '무작위 숫자 64개' 주사위 묶음이 '생성자' 상자로 들어가 '가짜 그림'이 나옵니다. 아래쪽에는 'MNIST 진짜 그림' 카드 묶음이 있습니다. 가짜 그림과 진짜 그림이 모두 오른쪽 '판별자' 상자로 들어가고, 판별자에서 '진짜일 확률(0~1)' 계기판이 나옵니다. 판별자에서 생성자로 돌아가는 점선 화살표에 '어디가 가짜 같은지(기울기)'가 적혀 있습니다.">


**실습 데이터:** 교안 01 본실습과 같은 손글씨 숫자 [MNIST](https://web.archive.org/web/20231231235959/http://yann.lecun.com/exdb/mnist/)(LeCun, Cortes & Burges, CC BY-SA 3.0) 표본입니다. GAN 학습에는 학습용 10,000장의 그림만 쓰고 정답 숫자는 쓰지 않습니다. 정답 숫자와 테스트용 2,000장은 5-4에서 생성된 숫자를 세는 분류기를 학습하고 확인할 때만 씁니다.

## 1. 실습 준비

생성자가 그릴 그림과 같은 모양으로 진짜 그림을 준비합니다.

### 1-1. 라이브러리와 데이터

준비 셀은 교안 01·02와 같습니다(계산 장치와 그래프 글꼴).

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 교안 01과 같은 손글씨 숫자 데이터입니다. GAN 학습에는 학습용 그림 10,000장만 씁니다
mnist = torch.load(data_dir / "본실습" / "mnist_small.pt")
print("학습용 그림:", mnist["train_images"].shape, mnist["train_images"].dtype)

### 1-2. 픽셀을 -1~1로 바꾸고 펼치기

생성자의 마지막 층은 Tanh(출력 -1~1, 앞 단원의 활성화 함수)이므로, 진짜 그림도 같은 범위로 맞춰야 판별자가 범위 차이만 보고 가짜를 가려내지 않습니다.

- **-1~1로 바꾸기:** 255로 나눠 0~1로 만든 뒤 2를 곱하고 1을 뺍니다. 바탕(0)은 -1, 중간(128)은 약 0, 가장 진한 획(255)은 1이 됩니다.
- **펼치기:** 생성자와 판별자가 `nn.Linear`로 된 신경망이므로 28 × 28 그림을 앞 단원의 DNN처럼 숫자 784개 한 줄로 펼칩니다.

In [ ]:
# 0~255 -> 0~1 -> -1~1 로 바꾸고, 그림마다 784개 한 줄로 펼칩니다
X_train = (mnist["train_images"].float() / 255) * 2 - 1
X_train = X_train.reshape(-1, 784)
y_train = mnist["train_labels"]  # 정답 숫자는 GAN 학습에 쓰지 않고, 5-4의 분류기 학습과 기대 개수에 씁니다

print("X_train:", X_train.shape, X_train.dtype)
print("픽셀 최솟값:", X_train.min().item(), "| 최댓값:", X_train.max().item())

In [ ]:
import math


def show_images(images, n_cols, title):
    """784개 한 줄인 그림들을 28 x 28로 되돌려 n_cols칸씩 격자로 그립니다."""
    images = images.detach().cpu()
    n_rows = math.ceil(len(images) / n_cols)  # 그림 수를 칸 수로 나눈 올림
    plt.figure(figsize=(n_cols * 1.0, n_rows * 1.0 + 0.6))
    for i in range(len(images)):
        plt.subplot(n_rows, n_cols, i + 1)
        # gray_r: -1(바탕)은 흰색, 1(획)은 검은색
        plt.imshow(images[i].reshape(28, 28), cmap="gray_r", vmin=-1, vmax=1)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# 생성자가 흉내 낼 진짜 그림 16장
show_images(X_train[:16], 16, "진짜 손글씨 숫자(학습 데이터)")

### 1-3. DataLoader

진짜 그림을 64장씩 섞어 꺼내도록 교안 01과 같은 `DataLoader`를 만듭니다. 미니배치마다 진짜 64장과 생성자가 만든 가짜 64장을 판별자에게 보여 줍니다.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# 학습용 그림 10,000장을 64장씩 섞어 꺼냅니다
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
print("에폭당 미니배치 수:", len(train_loader))

## 2. 생성자와 판별자 만들기

두 신경망은 지금까지 만든 DNN과 같은 `nn.Linear` 층의 묶음입니다. 다른 점은 입력과 출력의 모양뿐입니다.


<img src="images/91_generator_discriminator.png" width="1050" alt="제목은 '생성자와 판별자의 입력과 출력'입니다. 위쪽 분홍 띠 '생성자'에서는 '무작위 숫자 64개' 칸 묶음이 층 상자 '64', '256', '512', '784'와 'Tanh: -1~1'을 지나 손글씨 숫자 '28 x 28 그림 한 장'이 됩니다. 아래쪽 파랑 띠 '판별자'에서는 '28 x 28 그림 한 장(784개)'이 층 상자 '784', '512', '256', '1'을 지나 연노랑 상자 '로짓 하나'가 됩니다. 판별자 띠 바깥 아래의 점선 상자에 'Sigmoid -> 진짜일 확률(손실 안에서 계산)'이 있고, 로짓 상자에서 점선 화살표가 이어집니다.">


### 2-1. 생성자: 무작위 숫자 64개 -> 그림 한 장

- **잡음(noise):** 생성자의 입력인 무작위 숫자 64개입니다(잠재 벡터, latent vector). `torch.randn`으로 평균 0, 표준편차 1인 정규분포에서 뽑고, 잡음이 달라지면 다른 그림이 나옵니다.
- **층:** 64 -> 256 -> 512 -> 784로 넓혀 가며, 은닉층에는 ReLU를 씁니다.
- **마지막 Tanh:** 출력 784개를 -1~1로 맞춰, 앞에서 -1~1로 바꾼 진짜 그림과 같은 범위의 픽셀이 되게 합니다.

In [ ]:
import torch.nn as nn

noise_size = 64  # 잡음(무작위 숫자) 개수


def make_generator():
    """잡음 64개를 받아 -1~1 픽셀 784개(그림 한 장)를 내는 생성자를 만듭니다."""
    return nn.Sequential(
        nn.Linear(noise_size, 256),
        nn.ReLU(),
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, 784),
        nn.Tanh(),  # 픽셀 범위를 진짜 그림과 같은 -1~1로 맞춥니다
    )


# 학습 전 생성자에 잡음 8개를 넣어 그림 8장을 만듭니다
torch.manual_seed(0)
generator = make_generator().to(device)
noise = torch.randn(8, noise_size, device=device)
untrained_images = generator(noise)
print("잡음 모양:", noise.shape, "-> 생성한 그림 모양:", untrained_images.shape)
show_images(untrained_images, 8, "학습 전 생성자가 만든 그림")

학습 전 생성자는 파라미터가 무작위라 회색 잡음만 그립니다. 그림 8장의 모양은 (8, 784)로 진짜 그림 8장과 같습니다.

### 2-2. 판별자: 그림 한 장 -> 로짓 하나

- **이진 분류 모델:** 그림이 진짜(1)인지 가짜(0)인지 가리는 모델입니다. 앞 단원의 이진 분류 출력층과 손실 짝대로 출력은 로짓 하나이고, Sigmoid는 손실 안에 있으므로 모델 끝에 넣지 않습니다.
- **층:** 784 -> 512 -> 256 -> 1로 좁혀 갑니다.
- **LeakyReLU:** ReLU와 같지만 음수를 0으로 만들지 않고 0.2를 곱해 조금 남깁니다(-2 -> -0.4). 판별자의 기울기가 생성자에게 전해져야 생성자가 학습하므로, 기울기가 0이 되는 뉴런을 줄이려고 GAN의 판별자에서 흔히 씁니다.

In [ ]:
def make_discriminator():
    """그림 784픽셀을 받아 진짜다운 정도를 로짓 하나로 내는 판별자를 만듭니다."""
    return nn.Sequential(
        nn.Linear(784, 512),
        nn.LeakyReLU(0.2),  # 음수는 0.2를 곱해 남깁니다
        nn.Linear(512, 256),
        nn.LeakyReLU(0.2),
        nn.Linear(256, 1),  # 로짓 하나. Sigmoid는 손실 안에 있습니다
    )


# 학습 전 판별자에 진짜 4장과 가짜 4장을 넣고, 로짓을 진짜일 확률로 바꿉니다
discriminator = make_discriminator().to(device)
real_logits = discriminator(X_train[:4].to(device))
fake_logits = discriminator(untrained_images[:4])
untrained_table = pd.DataFrame({
    "진짜 그림: 진짜일 확률": torch.sigmoid(real_logits).flatten().tolist(),
    "가짜 그림: 진짜일 확률": torch.sigmoid(fake_logits).flatten().tolist(),
}).round(3)
display(untrained_table)

학습 전 판별자는 진짜와 가짜 모두에 약 0.5(반반)를 줍니다. 아직 둘을 가릴 줄 모릅니다.

## 3. 두 신경망의 목표: 이진 분류 손실

두 신경망 모두 판별자의 로짓 하나에 앞 단원의 `nn.BCEWithLogitsLoss`(출력 1개 + 정답 0.0 또는 1.0)를 씁니다. 같은 손실에 정답만 다르게 줍니다.

- 그림 $x$를 판별자에 넣어 나온 로짓을 Sigmoid에 통과시킨 값을 $D(x)$(진짜일 확률), 생성자가 잡음 $z$로 만든 그림을 $G(z)$라 합니다.
- **판별자의 목표:** 진짜 그림은 정답 1(진짜), 가짜 그림은 정답 0(가짜)으로 맞힙니다. 판별자 손실은 진짜 손실과 가짜 손실의 합입니다.

$\displaystyle L_D = -\log D(x) - \log\bigl(1 - D(G(z))\bigr)$

- **생성자의 목표:** 자기가 만든 가짜 그림을 판별자가 1(진짜)로 보게 만듭니다. 그래서 가짜 그림에 정답 1을 주고 손실을 계산합니다.

$\displaystyle L_G = -\log D(G(z))$

- **정답 1을 주는 이유:** 판별자가 가짜를 쉽게 가려내는 학습 초반에도 생성자 손실이 커서 기울기가 작아지지 않습니다. 그래서 생성자가 처음부터 배울 수 있습니다([Goodfellow 외, 2014](https://arxiv.org/abs/1406.2661)).


<img src="images/92_gan_loss_targets.png" width="1050" alt="제목은 '같은 손실, 다른 정답'입니다. 가운데에 분홍 테두리의 가짜 숫자 3 그림이 '판별자'로 들어가 '로짓 2.0 (진짜일 확률 0.88)'이 나옵니다. 왼쪽 파랑 판 '판별자의 손실'에는 '정답 0 (가짜)', '손실 2.13', '크다: 판별자는 이 로짓을 낮추도록 갱신'이 적혀 있습니다. 오른쪽 분홍 판 '생성자의 손실'에는 '정답 1 (진짜로 속이기)', '손실 0.13', '작다: 생성자는 이미 잘 속이고 있음'이 적혀 있습니다. 아래 띠에 'nn.BCEWithLogitsLoss: 진짜 = 1, 가짜 = 0'이 적혀 있습니다.">


판별자가 가짜 그림 한 장에 로짓 2.0을 줬다고 하면(아래 셀과 같은 숫자입니다):

1. **진짜일 확률:** Sigmoid로 0.88입니다. 판별자가 가짜를 진짜로 착각했습니다.
2. **판별자 손실(정답 0):** $-\log(1 - 0.88) \approx 2.13$으로 큽니다. 판별자는 이 그림의 로짓을 낮추도록 갱신됩니다.
3. **생성자 손실(정답 1):** $-\log 0.88 \approx 0.13$으로 작습니다. 생성자는 이미 판별자를 잘 속이고 있습니다.

In [ ]:
# 가짜 그림 하나에 판별자가 로짓 2.0을 줬을 때, 정답을 다르게 준 두 손실
# BCE = 평균(-정답 * log(p) - (1 - 정답) * log(1 - p)), p = 진짜일 확률
loss_fn = nn.BCEWithLogitsLoss()
logit = torch.tensor([2.0])
print("진짜일 확률:", round(torch.sigmoid(logit).item(), 2))
print("판별자 손실(정답 0, 가짜):", round(loss_fn(logit, torch.tensor([0.0])).item(), 2))
print("생성자 손실(정답 1, 진짜로 속이기):", round(loss_fn(logit, torch.tensor([1.0])).item(), 2))

## 4. 번갈아 학습하기

두 신경망은 옵티마이저를 하나씩 따로 두고, 미니배치마다 판별자를 한 번 갱신한 뒤 생성자를 한 번 갱신합니다.


<img src="images/93_gan_training_step.png" width="1050" alt="제목은 '미니배치 하나에서 두 번 갱신합니다'입니다. 세 칸이 나란히 있습니다. '1 가짜 만들기': '잡음 64개 x 64장'이 '생성자'를 지나 '가짜 64장'이 됩니다. '2 판별자 갱신': '진짜 64장'은 판별자를 지나 '정답 1', '가짜 64장'은 판별자를 지나 '정답 0'과 비교되고, 자물쇠가 걸린 생성자 옆에 'detach: 기울기가 생성자로 가지 않음', 아래에 '판별자 손실 = 진짜 손실 + 가짜 손실'이 있습니다. '3 생성자 갱신': '가짜 64장(1단계와 같은 것)'이 판별자를 지나 '정답 1'과 비교되고, 자물쇠가 걸린 판별자 옆에 '판별자는 그대로(g_optimizer는 생성자만 갱신)', 아래에 '생성자 손실'이 있습니다. 아래 띠에 '157번 반복 = 1에폭'이 적혀 있습니다.">


1. **가짜 만들기:** 잡음 64개짜리 64묶음을 생성자에 넣어 가짜 그림 64장을 만듭니다.
2. **판별자 갱신:** 진짜 64장은 정답 1, 가짜 64장은 정답 0으로 손실을 더해 판별자만 갱신합니다. 가짜 그림에 `.detach()`를 붙여 이 손실의 기울기가 생성자까지 가지 않게 합니다. `.detach()`를 빼면 판별자 손실의 `backward()`가 생성자 쪽 계산 기록까지 지워, 3단계의 `backward()`에서 오류가 납니다.
3. **생성자 갱신:** 같은 가짜 64장을 판별자에 다시 넣고 정답 1로 손실을 계산합니다. 기울기는 판별자를 거쳐 생성자까지 가지만, `g_optimizer`가 생성자의 파라미터만 갱신합니다.
4. 미니배치 157개를 다 쓰면 한 에폭입니다.

**Adam 설정**
- 학습률 0.0002, `betas=(0.5, 0.999)`는 합성곱 층을 쓰는 GAN인 DCGAN(deep convolutional GAN) 논문(Radford 외, 2015)이 쓴 값입니다.
- `betas`의 첫 값은 모멘텀처럼 이전 기울기를 얼마나 이어 갈지 정합니다. 논문은 기본값 0.9에서 학습이 흔들려, 학습을 안정시키려고 0.5로 낮췄습니다.

In [ ]:
def train_gan_one_epoch(generator, discriminator, loader, loss_fn, g_optimizer, d_optimizer):
    """미니배치마다 판별자와 생성자를 한 번씩 갱신하고, 두 손실의 에폭 평균을 돌려줍니다."""
    d_total = 0.0
    g_total = 0.0
    for real_images, _ in loader:  # 정답 숫자는 쓰지 않아 _로 받습니다
        real_images = real_images.to(device)
        batch_size = len(real_images)
        real_targets = torch.ones(batch_size, 1, device=device)  # 진짜 정답 1
        fake_targets = torch.zeros(batch_size, 1, device=device)  # 가짜 정답 0

        # 잡음으로 가짜 그림을 만듭니다
        noise = torch.randn(batch_size, noise_size, device=device)
        fake_images = generator(noise)

        # 판별자 갱신: 진짜는 1, 가짜는 0으로 맞히게 합니다
        d_optimizer.zero_grad()
        real_loss = loss_fn(discriminator(real_images), real_targets)
        fake_loss = loss_fn(discriminator(fake_images.detach()), fake_targets)  # detach: 기울기가 생성자로 가지 않게
        d_loss = real_loss + fake_loss  # 판별자 손실 = 진짜 손실 + 가짜 손실
        d_loss.backward()
        d_optimizer.step()

        # 생성자 갱신: 판별자가 가짜를 1(진짜)로 보게 합니다. 생성자 손실 = 가짜에 정답 1을 준 손실
        g_optimizer.zero_grad()
        g_loss = loss_fn(discriminator(fake_images), real_targets)
        g_loss.backward()
        g_optimizer.step()

        d_total += d_loss.item()
        g_total += g_loss.item()

    return d_total / len(loader), g_total / len(loader)

학습하면서 같은 잡음 16개로 그린 그림을 몇 에폭마다 저장해 두면, 생성자가 나아지는 과정을 같은 자리에서 비교할 수 있습니다.

- **에폭:** 150에폭 학습합니다. CPU에서 3분 안팎 걸립니다(25에폭마다 진행 상황을 출력합니다).
- **저장할 때:** 학습 전과 1, 10, 25, 50, 100, 150에폭에서 고정 잡음 16개로 그린 그림을 저장합니다.
- **실행마다 다른 결과:** 계산 장치와 라이브러리 버전에 따라 손실과 그림이 달라집니다. 결과는 숫자 하나하나보다 흐름의 경향으로 읽습니다.

In [ ]:
# 학습 전 상태에서 시작하도록 시드를 고정하고 두 모델과 옵티마이저를 새로 만듭니다
torch.manual_seed(0)  # 모델 초깃값, 잡음, 미니배치 섞는 순서를 고정합니다
generator = make_generator().to(device)
discriminator = make_discriminator().to(device)
g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# 학습 내내 같은 잡음 16개로 그림을 그려 에폭마다 비교합니다
fixed_noise = torch.randn(16, noise_size, device=device)
n_epochs = 150
snapshot_epochs = [1, 10, 25, 50, 100, 150]

In [ ]:
import time

# 에폭마다 두 손실을 기록하고, 정해 둔 에폭에서 고정 잡음으로 그린 그림을 저장합니다
history_rows = []
with torch.no_grad():
    snapshots = {0: generator(fixed_noise).cpu()}  # 0: 학습 전 상태
start_time = time.time()
for epoch in range(1, n_epochs + 1):
    d_loss, g_loss = train_gan_one_epoch(generator, discriminator, train_loader, loss_fn, g_optimizer, d_optimizer)
    history_rows.append({"에폭": epoch, "판별자 손실": d_loss, "생성자 손실": g_loss})

    if epoch in snapshot_epochs:
        with torch.no_grad():
            snapshots[epoch] = generator(fixed_noise).cpu()

    # 25에폭마다 진행 상황을 출력합니다
    if epoch % 25 == 0:
        elapsed = time.time() - start_time
        print(f"에폭 {epoch:3d} | 판별자 손실 {d_loss:.3f} | 생성자 손실 {g_loss:.3f} | {elapsed:.0f}초")

history_table = pd.DataFrame(history_rows)

## 5. 결과 보기

### 5-1. 학습 전과 학습 후 비교

한 줄이 한 시점이고, 같은 열은 같은 잡음입니다. 맨 위 줄이 학습 전 생성자의 출력, 맨 아래 줄이 150에폭 학습 뒤의 출력입니다.

In [ ]:
# 행: 학습 전(0)과 저장한 에폭, 열: 고정 잡음 16개
shown_epochs = [0] + snapshot_epochs
n_rows = len(shown_epochs)
plt.figure(figsize=(16, n_rows * 1.15))
for row, epoch in enumerate(shown_epochs):
    for col in range(16):
        plt.subplot(n_rows, 16, row * 16 + col + 1)
        plt.imshow(snapshots[epoch][col].reshape(28, 28), cmap="gray_r", vmin=-1, vmax=1)
        plt.xticks([])
        plt.yticks([])
        if col == 0:
            if epoch == 0:
                row_name = "학습 전"
            else:
                row_name = f"{epoch}에폭"
            plt.ylabel(row_name, rotation=0, labelpad=30, fontsize=11)
plt.suptitle("같은 잡음 16개로 그린 그림: 학습 전과 에폭별 비교", fontsize=13)
plt.tight_layout()
plt.show()

**결과 읽기**
- **학습 전:** 16칸 모두 회색 잡음이고 숫자의 흔적이 없습니다. 파라미터가 무작위라 픽셀이 무작위로 정해진 상태입니다.
- **1에폭:** 16칸이 거의 같은 흐린 얼룩입니다. 잡음이 달라도 같은 그림이 나와, 생성자가 아직 잡음의 차이를 그림에 반영하지 못합니다.
- **10~50에폭:** 바탕이 하얘지고 획이 가운데에 모입니다. 점이 흩어져 알아보기 어려운 칸이 많고, 세로 획(1과 비슷한 모양)이 늘어납니다.
- **100·150에폭:** 칸마다 다른 숫자가 보이고, 150에폭의 획이 더 또렷합니다.

### 5-2. 두 손실 곡선

두 손실은 서로 겨루는 상대의 실력에 따라 바뀌므로, 분류 모델처럼 0을 향해 줄어들지 않습니다.

- **비길 때의 값:** 판별자가 진짜와 가짜를 전혀 못 가려 모두에 0.5를 주면, 판별자 손실은 $-\log 0.5 - \log 0.5 \approx 1.39$(진짜 손실 + 가짜 손실), 생성자 손실은 $-\log 0.5 \approx 0.69$입니다. 그래프에 점선으로 함께 그립니다.

In [ ]:
# 에폭별 두 손실과, 판별자가 모두에 0.5를 줄 때(비길 때)의 손실
plt.figure(figsize=(10, 4))
plt.plot(history_table["에폭"], history_table["판별자 손실"], label="판별자 손실")
plt.plot(history_table["에폭"], history_table["생성자 손실"], label="생성자 손실")
plt.axhline(2 * math.log(2), color="tab:blue", linestyle="--", linewidth=1, label="비길 때 판별자 손실 1.39")
plt.axhline(math.log(2), color="tab:orange", linestyle="--", linewidth=1, label="비길 때 생성자 손실 0.69")
plt.xlabel("에폭")
plt.ylabel("손실")
plt.title("판별자 손실과 생성자 손실")
plt.legend(loc="upper right")
plt.show()

# 그림을 저장한 에폭의 두 손실
snapshot_loss_table = history_table[history_table["에폭"].isin(snapshot_epochs)].set_index("에폭").round(3)
display(snapshot_loss_table)

**결과 읽기**
- **1에폭:** 두 손실이 비슷합니다.
- **초반:** 생성자 손실은 몇 에폭 만에 크게 올라가고, 판별자 손실은 낮게 머뭅니다. 판별자가 가짜를 쉽게 가려내는 구간입니다.
- **중반:** 생성자 손실은 오르내리며 내려오고, 판별자 손실은 천천히 올라옵니다.
- **후반(100에폭 뒤):** 두 손실이 거의 평평합니다. 같은 잡음으로 그린 그림도 이 무렵 대부분 숫자 모양입니다.
- **비길 때와 비교:** 150에폭의 판별자 손실은 비길 때(1.39)보다 낮아, 판별자가 여전히 조금 앞서 있습니다. 손실 숫자만으로는 그림이 좋아졌는지 알 수 없으므로 그림을 함께 봅니다.

학습 전에 본 판별자 표를 같은 진짜 그림 4장과 같은 잡음 4개로 다시 만들어, 학습한 판별자가 둘을 가리는지 봅니다.

In [ ]:
# 학습 전 표와 같은 진짜 4장, 같은 잡음 4개로 학습한 생성자가 만든 가짜 4장
with torch.no_grad():
    real_logits = discriminator(X_train[:4].to(device))
    fake_logits = discriminator(generator(noise[:4]))

# 학습한 판별자의 로짓을 진짜일 확률로 바꿉니다
trained_table = pd.DataFrame({
    "진짜 그림: 진짜일 확률": torch.sigmoid(real_logits).flatten().tolist(),
    "가짜 그림: 진짜일 확률": torch.sigmoid(fake_logits).flatten().tolist(),
}).round(3)
display(trained_table)

In [ ]:
# 4장은 그림마다 차이가 커서, 진짜 1,000장과 가짜 1,000장의 평균 확률도 구합니다
with torch.no_grad():
    real_probs = torch.sigmoid(discriminator(X_train[:1000].to(device)))
    fake_probs = torch.sigmoid(discriminator(generator(torch.randn(1000, noise_size, device=device))))
print(f"진짜 1,000장의 평균 확률: {real_probs.mean().item():.3f}")
print(f"가짜 1,000장의 평균 확률: {fake_probs.mean().item():.3f}")

**결과 읽기**
- **4장:** 학습 전에는 진짜와 가짜 모두 0.5 근처였지만, 학습 뒤에는 0.5에서 멀어진 값이 섞입니다. 그림마다 차이가 커서 4장만으로는 판단하기 어렵습니다.
- **1,000장 평균:** 진짜 그림의 평균 확률이 가짜 그림보다 높습니다. 학습 전과 달리 판별자가 진짜와 가짜를 어느 정도 가립니다.
- **0과 1까지는 벌어지지 않음:** 두 평균이 모두 0과 1에서 멀어, 생성자가 만든 가짜가 판별자를 일부 속입니다. 판별자 손실이 0으로 줄지 않는 것과 같은 이유입니다.

### 5-3. 새 잡음으로 새 숫자 만들기

학습을 마친 생성자는 잡음만 바꾸면 계속 새 그림을 만듭니다. 처음 보는 잡음 64개로 64장을 만듭니다.

In [ ]:
# 새 잡음 64개로 그림 64장을 만듭니다. 학습이 아니므로 기울기는 계산하지 않습니다
torch.manual_seed(1)
new_noise = torch.randn(64, noise_size, device=device)
with torch.no_grad():
    new_images = generator(new_noise)
show_images(new_images, 16, "학습한 생성자가 새 잡음으로 만든 숫자 64장")

**결과 읽기**
- 64장 대부분이 알아볼 수 있는 숫자입니다. 잡음만 바꿨는데 숫자 종류, 굵기, 기울기가 다른 그림이 나옵니다.
- 획 둘레에 점이 흩어져 있거나 알아보기 어려운 그림도 섞여 있습니다. 이 부록은 `nn.Linear` 층만 썼고, 생성자와 판별자에 합성곱 층을 쓰는 GAN(DCGAN)은 이보다 선명한 그림을 만듭니다.

### 5-4. 모드 붕괴 확인: 숫자 종류별 개수

**모드 붕괴(mode collapse):** 생성자가 만드는 그림이 학습 데이터에 있는 종류(모드) 일부로만 몰리고, 나머지 종류는 거의 만들지 않는 현상입니다. 이 데이터에서 종류는 숫자 0~9입니다.

- 그림 16장에서 같은 숫자가 겹치는 것은 증거가 아닙니다. 숫자가 10종이라 고르게 만들어도 겹침이 생깁니다.
- 그래서 많은 그림의 숫자를 분류기로 읽어 종류별 개수를 셉니다. 고르게 만든다면 학습 데이터의 숫자 비율에 맞춰 개수가 나옵니다.
- 분류기는 학습용 그림 10,000장과 정답 숫자로 `nn.Linear` 층만 쌓아 짧게 학습하고, 테스트용 그림 2,000장으로 정확도를 확인합니다.

In [ ]:
# 생성한 그림의 숫자를 읽을 분류기: 784 -> 128 -> 10, 로짓 10개를 내는 DNN
torch.manual_seed(0)
classifier = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
c_optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
c_loss_fn = nn.CrossEntropyLoss()

# 5에폭 학습합니다. 입력은 GAN과 같은 -1~1 픽셀 784개입니다
for epoch in range(5):
    for images, labels in train_loader:
        c_optimizer.zero_grad()
        c_loss = c_loss_fn(classifier(images.to(device)), labels.to(device))
        c_loss.backward()
        c_optimizer.step()

# 테스트용 그림 2,000장으로 정확도를 확인합니다
X_test = (mnist["test_images"].float() / 255) * 2 - 1
with torch.no_grad():
    test_predictions = classifier(X_test.reshape(-1, 784).to(device)).argmax(dim=1).cpu()
print(f"분류기 테스트 정확도: {(test_predictions == mnist['test_labels']).float().mean().item():.3f}")

In [ ]:
# 새 잡음 2,000개로 그림 2,000장을 만들고, 분류기가 읽은 숫자별 개수를 셉니다
torch.manual_seed(2)
with torch.no_grad():
    generated = generator(torch.randn(2000, noise_size, device=device))
    generated_digits = classifier(generated).argmax(dim=1).cpu()

# 기대 개수 = 학습 데이터의 숫자 비율 * 2,000. 고르게 만들어도 우연한 차이가 약 30 있습니다
generated_counts = torch.bincount(generated_digits, minlength=10)
train_share = torch.bincount(y_train, minlength=10) / len(y_train)
digit_table = pd.DataFrame({
    "숫자": range(10),
    "생성한 그림 개수": generated_counts.tolist(),
    "기대 개수": (train_share * 2000).round().int().tolist(),
}).set_index("숫자")
display(digit_table)

In [ ]:
# 숫자별 생성 개수(막대)와 기대 개수(점선)
plt.figure(figsize=(8, 3.5))
plt.bar(digit_table.index, digit_table["생성한 그림 개수"], label="생성한 그림 개수")
plt.plot(digit_table.index, digit_table["기대 개수"], color="tab:red", linestyle="--", marker="o", label="기대 개수")
plt.xticks(range(10))
plt.xlabel("분류기가 읽은 숫자")
plt.ylabel("그림 수")
plt.title("생성한 그림 2,000장의 숫자별 개수")
plt.legend()
plt.show()

**결과 읽기**
- **분류기 정확도:** 테스트용 그림에서 약 0.92이므로, 생성한 그림 중 알아보기 어려운 그림은 엉뚱한 숫자로 읽힐 수 있습니다. 개수는 대략적인 분포로 읽습니다.
- **쏠림:** 숫자 1이 기대 개수의 두 배 이상이고, 7과 9도 기대보다 많습니다. 0, 2, 5, 6은 기대보다 크게 적습니다. 고르게 만들어도 생기는 우연한 차이(약 30)보다 훨씬 큽니다.
- **완전한 붕괴는 아님:** 열 숫자가 모두 80개 넘게 나옵니다. 한두 종류로만 몰리지는 않았지만 숫자 종류의 비율이 학습 데이터와 다릅니다. 5-1의 맨 아래 줄에서 세로 획 1이 여럿인 것과 맞습니다.
- 그래서 GAN은 손실 곡선만 보지 않고, 만든 그림의 종류별 개수까지 확인합니다.

## 6. 핵심 정리

- **GAN:** 생성자(잡음 -> 그림)와 판별자(그림 -> 진짜다운 정도를 나타내는 로짓)가 겨루며 함께 학습하는 생성 모델입니다. 분류 모델이 그림에서 답을 고른다면, 생성자는 잡음에서 그림을 만듭니다.
- **같은 손실, 다른 정답:** 두 신경망 모두 `nn.BCEWithLogitsLoss`를 씁니다. 판별자는 진짜 1·가짜 0을 맞히고, 생성자는 가짜에 정답 1을 붙여 판별자를 속입니다.
- **번갈아 학습:** 미니배치마다 판별자를 한 번(가짜는 `.detach()`), 생성자를 한 번 갱신합니다. 옵티마이저는 신경망마다 따로 둡니다.
- **범위 맞추기:** 생성자 끝의 Tanh(-1~1)에 맞춰 진짜 그림도 -1~1로 바꿉니다.
- **학습 전후 비교:** 같은 잡음 16개의 출력이 회색 잡음에서 숫자 모양으로 바뀝니다. 두 손실은 0으로 줄지 않고 서로 밀고 당기므로, 학습 상태는 손실보다 그림으로 판단합니다.
- **모드 붕괴:** 생성자가 학습 데이터의 일부 종류로만 몰리는 현상입니다. 생성한 그림 2,000장을 분류기로 읽어 숫자별 개수를 기대 개수와 비교해 확인합니다.